# The Lorenz equations

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZetaOne/pyauto-xpp/blob/main/tutorials/notebooks/07_lorenz.ipynb)

The classic chaotic flow. For r = 27 the orbit switches irregularly between the two
wings; for r below about 24.74 the two non-trivial equilibria are stable.

*Based on an example from XPP Studio (Peter Ashwin and Bard Ermentrout, GPL-3.0). Every number below is computed by XPPAUT 8.2 and its AUTO through pyauto-xpp.*

Running in Google Colab? This cell installs pyauto-xpp first; elsewhere it does nothing.

In [1]:
import sys
if "google.colab" in sys.modules:
    %pip -q install pyauto-xpp "plotly>=6" anywidget ipywidgets --find-links https://github.com/ZetaOne/pyauto-xpp/releases/expanded_assets/v0.3.0
    import urllib.request
    urllib.request.urlretrieve("https://raw.githubusercontent.com/ZetaOne/pyauto-xpp/main/tutorials/notebooks/xpptut.py", "xpptut.py")
    from google.colab import output
    output.enable_custom_widget_manager()

In [2]:
import plotly.graph_objects as go
from pyxpp.builder import ModelBuilder
from xpptut import Explorer, load, orbit_diagram, show_figure

## The model

Written with `ModelBuilder`, statement by statement as in an `.ode` file. Edit it and rerun the cells below.

In [3]:
mb = (ModelBuilder("lorenz")
      .ode("x", "s*(-x+y)")
      .ode("y", "r*x-y-x*z")
      .ode("z", "-b*z+x*y")
      .parameter(r=27, s=10, b=2.666667)
      .initial(x=-7.5, y=-3.6, z=30)
      .option(total=40, dt=0.01, meth="rk4", bound=1000))
print(mb.render())

x'=s*(-x+y)
y'=r*x-y-x*z
z'=-b*z+x*y
par r=27,s=10,b=2.666667
init x=-7.5,y=-3.6,z=30
@ total=40,dt=0.01,meth=rungekutta,bound=1000
done



## Explore

The sliders, the direction-field menu and the AUTO panel all call XPP. `ex.m` is the pyxpp model, for anything else.

In [4]:
ex = Explorer(mb, kind="3d", x="x", y="y", z="z", sliders={"r": (0, 40), "s": (1, 20)}, show=["x", "z"],
              auto="r", auto_range=(0, 40), title="Lorenz")
ex

(run this cell with the kernel to get the interactive explorer)

## Going further

**Equilibria in `r`**: the pitchfork at r = 1 and the Hopf point near 24.74.

In [5]:
from pyxpp.builder import ModelBuilder
m = load(mb)
m.set(r=0.5); m.run(total=200, x=0.1, y=0.1, z=0.1); m.run_last(total=10)
d = m.auto.run("r", start="steady", ds=0.02, dsmax=0.2, parmin=0, parmax=40, nmax=500)
print(sorted({(p.type, round(p.x, 4)) for p in d.labeled() if p.type in ("BP", "HB")}))

[('BP', 1.0), ('HB', 24.7369)]


**Sensitive dependence**: two starts 1e-8 apart.

In [6]:
import numpy as np
a = ex.m.run(keep=False, total=40); b = ex.m.run(keep=False, total=40, x=-7.5 + 1e-8)
show_figure(go.Figure(go.Scatter(x=a.t, y=np.abs(a["x"] - b["x"]) + 1e-12)), xaxis_title="t",
            yaxis_title="|difference in x|", yaxis_type="log", height=300)

## Things to try

- Set `r` = 20: the orbit spirals into one of the two equilibria.